In [1]:
import os
with open(r"C:\Users\silas\openai.txt", "r") as f:
    api_key = f.read().strip()
os.environ["OPENAI_API_KEY"] = api_key

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
df = pd.read_csv(r"./ground_truth_final.csv")
df = df.astype(object).where(pd.notna(df), None)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df_sample = train_df.sample(5, random_state=42)

In [3]:
from nps_crawling.classification.categories.category import ClassificationProperty, ClassificationType, Example

nps_category_properties = [
    ClassificationProperty(
        name="KPI_CURRENT_VALUE",
        description="The text reports a specific NPS value.",
        type=ClassificationType.BOOLEAN,
    ),
    ClassificationProperty(
        name="KPI_TREND",
        description="The text describes change over time.",
        type=ClassificationType.BOOLEAN,
    ),
    ClassificationProperty(
        name="KPI_HISTORICAL_COMPARISON",
        description="The text explicitly compares NPS against a previous period.",
        type=ClassificationType.BOOLEAN,
    ),
    ClassificationProperty(
        name="TARGET_OUTLOOK",
        description="The text expresses a future goal, target, ambition, forecast, or expected future NPS.",
        type=ClassificationType.BOOLEAN,
    ),
    ClassificationProperty(
        name="NPS_GOAL_REACHED",
        description="The text states that an NPS target or objective has been met or exceeded.",
        type=ClassificationType.BOOLEAN,
    ),
    ClassificationProperty(
        name="METHODOLOGY_DEFINITION",
        description="The text explains what NPS is, how it is calculated, or what it measures.",
        type=ClassificationType.BOOLEAN,
    ),
    ClassificationProperty(
        name="QUALITATIVE_ONLY",
        description="NPS is mentioned meaningfully but none of the other labels apply.",
        type=ClassificationType.BOOLEAN,
    ),
]

nps_value_category_properties = [
    ClassificationProperty(
        name="nps_value_fix",
        description="Direct NPS value.",
        type=ClassificationType.FLOAT,
    ),
    ClassificationProperty(
        name="nps_competition_industry",
        description="Industry benchmark or competitor NPS value.",
        type=ClassificationType.FLOAT,
    ),
    ClassificationProperty(
        name="nps_value_over",
        description="Threshold value associated with above, over, greater than, more than, or at least.",
        type=ClassificationType.FLOAT,
    ),
    ClassificationProperty(
        name="nps_value_below",
        description="Threshold value associated with below, under, less than, or at most.",
        type=ClassificationType.FLOAT,
    ),
    ClassificationProperty(
        name="nps_goal_value",
        description="Explicit NPS target value, only if value is at least 20.",
        type=ClassificationType.FLOAT,
    ),
    ClassificationProperty(
        name="nps_goal_change",
        description="Planned improvement amount such as improve by 10, increase by 5, or raise by 7.",
        type=ClassificationType.FLOAT,
    ),
]

has_numeric_nps_property = ClassificationProperty(
    name="has_numeric_nps",
    description="True if the text contains at least one explicit numeric NPS value.",
    type=ClassificationType.BOOLEAN,
)

all_classification_properties = [
    *nps_category_properties,
    *nps_value_category_properties,
    has_numeric_nps_property,
]

In [1]:
from nps_crawling.classification.categories.category import ClassificationProperty, ClassificationType, Example, DataEntry
examples = [Example(
    text = "Input text one",
    answer=[DataEntry("prop1.name", "value"), DataEntry("prop2.name", "value")]
),
    Example(
    text = "Input text two",
    answer=[DataEntry("prop1.name", "value"), DataEntry("prop2.name", "value")]
)]

default_properties = [
    ClassificationProperty(
        name="prop1.name",
        description="prop1.description",
        type=ClassificationType.BOOLEAN,
    ),
    ClassificationProperty(
        name="prop2.name",
        description="prop2.description",
        type=ClassificationType.BOOLEAN,
    ),
]

prompt_base = "category.prompt_base"
from nps_crawling.classification.categories.category import ClassificationCategory
category = ClassificationCategory("Default", default_properties, prompt_base, examples=examples)

In [1]:
from nps_crawling.classification.model_pipeline import ClassificationModelPipeline
import json
from pathlib import Path

config_path = Path(r"C:\Users\silas\VSProjects\NPS-Crawling\src\nps_crawling\classification\configurations\Default.json")
with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

pipeline = ClassificationModelPipeline.from_dict(config)

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [8]:
from nps_crawling.classification.model_pipeline import ClassificationModelPipeline
import os
os.environ["LOCAL_MODE"] = "0"
pipeline = ClassificationModelPipeline("NPS_ALL_WITH_QWEN", classification_configuration={
    r"C:\Users\silas\VSProjects\NPS-Crawling\src\nps_crawling\classification\configurations\categories\NPS All\fe4b8984bb746616f24c832fd83022578959197cc07bbb4cc3d889bc5bc0fac1.json":
    r"C:\Users\silas\VSProjects\NPS-Crawling\src\nps_crawling\classification\configurations\Qwen3-8B\fc71e4662a1dee0f734b09d3a2d746c791b2ad11156266b7b59b75cb9e433773.json"})

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

In [7]:
from nps_crawling.classification.models.registry import ClassificationModelName, get_model
from nps_crawling.classification.categories.category import ClassificationCategory
prompt_base = """You are an expert at analyzing text to identify and extract information about Net Promoter Score (NPS). NPS is a widely used metric that measures customer loyalty and satisfaction by asking customers how likely they are to recommend a product or service to others. Your task is to read the provided text and determine whether it contains specific information related to NPS, as defined by the following properties:"""
category = ClassificationCategory("NPS All", all_classification_properties, prompt_base)

In [3]:
from nps_crawling.classification.models.registry import ClassificationModelName, get_model
from nps_crawling.classification.categories.registry import ClassificationTask, get_category

# Example usage
model = get_model(ClassificationModelName.OPENAI, model_name="gpt-5.4-mini")
category_1 = get_category(ClassificationTask.NPS_CATEGORY)
category_2 = get_category(ClassificationTask.HAS_NUMERIC_NPS)
category_3 = get_category(ClassificationTask.NPS_VALUE_CATEGORY)

In [7]:
model.train(train_df, category)

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)
results_1 = model.evaluate(test_df, category_1)
results_2 = model.evaluate(test_df, category_2)
results_3 = model.evaluate(test_df, category_3)

INFO:nps_crawling.classification.models.model:Classifying text 1/141
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:nps_crawling.classification.models.model:Classifying text 2/141
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:nps_crawling.classification.models.model:Classifying text 3/141
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:nps_crawling.classification.models.model:Classifying text 4/141
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:nps_crawling.classification.models.model:Classifying text 5/141
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:nps_crawling.classification.models.model:Classifying text 6/141
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:nps_crawling.classification.models.

: 